# NLP Lab Task 5
N-gram language models using the Python standard library.

## Task 1 — Tokenization

In [1]:
train = ["the cat sat on the mat", "the cat ate the fish",
         "the dog sat on the rug", "the dog ate the bone"]
test = ["the cat sat on the rug", "the fish sat on the mat"]

def tokenize(sentence):
    return ["<s>", *sentence.lower().split(), "</s>"]

def build_vocab(sentences):
    return {token for sentence in sentences for token in tokenize(sentence) if token != "<s>"}

train_tokens = [tokenize(sentence) for sentence in train]
test_tokens = [tokenize(sentence) for sentence in test]
vocab = build_vocab(train)
V = len(vocab)
for label, corpus in (("Train", train_tokens), ("Test", test_tokens)):
    for tokens in corpus:
        print(label, tokens)
print("Training tokens with markers:", sum(map(len, train_tokens)))
print("Training tokens without markers:", sum(len(tokens) - 2 for tokens in train_tokens))
print("V:", V)
print("Vocabulary:", sorted(vocab))

Train ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'mat', '</s>']
Train ['<s>', 'the', 'cat', 'ate', 'the', 'fish', '</s>']
Train ['<s>', 'the', 'dog', 'sat', 'on', 'the', 'rug', '</s>']
Train ['<s>', 'the', 'dog', 'ate', 'the', 'bone', '</s>']
Test ['<s>', 'the', 'cat', 'sat', 'on', 'the', 'rug', '</s>']
Test ['<s>', 'the', 'fish', 'sat', 'on', 'the', 'mat', '</s>']
Training tokens with markers: 30
Training tokens without markers: 22
V: 11
Vocabulary: ['</s>', 'ate', 'bone', 'cat', 'dog', 'fish', 'mat', 'on', 'rug', 'sat', 'the']


## Task 2 — Generating N-grams

In [2]:
def get_ngrams(tokens, n):
    if n < 1:
        raise ValueError("n must be at least 1")
    return [tuple(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

for n in (1, 2, 3):
    print(f"{n}-grams:", get_ngrams(train_tokens[0], n))
for tokens in train_tokens + test_tokens:
    for n in (1, 2, 3):
        assert len(get_ngrams(tokens, n)) == len(tokens) - n + 1
print("Length formula verified for every sentence.")
print("n larger than length:", get_ngrams(train_tokens[0], len(train_tokens[0]) + 1))

1-grams: [('<s>',), ('the',), ('cat',), ('sat',), ('on',), ('the',), ('mat',), ('</s>',)]
2-grams: [('<s>', 'the'), ('the', 'cat'), ('cat', 'sat'), ('sat', 'on'), ('on', 'the'), ('the', 'mat'), ('mat', '</s>')]
3-grams: [('<s>', 'the', 'cat'), ('the', 'cat', 'sat'), ('cat', 'sat', 'on'), ('sat', 'on', 'the'), ('on', 'the', 'mat'), ('the', 'mat', '</s>')]
Length formula verified for every sentence.
n larger than length: []


## Task 3 — Counting N-grams

In [3]:
from collections import Counter

counts = {n: Counter(gram for tokens in train_tokens for gram in get_ngrams(tokens, n))
          for n in (1, 2, 3)}

def count_of(ngram):
    return counts[len(ngram)][ngram]

for n in (1, 2, 3):
    print(f"\n{n}-gram counts:")
    for gram, count in sorted(counts[n].items(), key=lambda item: (-item[1], item[0])):
        print(gram, count)
print("\nTop 5 bigrams:", sorted(counts[2].items(), key=lambda item: (-item[1], item[0]))[:5])
print("Distinct counts:", {n: len(counts[n]) for n in (1, 2, 3)})
print("Seen:", count_of(("the", "cat")), "Unseen:", count_of(("fish", "sat")))


1-gram counts:
('the',) 8
('</s>',) 4
('<s>',) 4
('ate',) 2
('cat',) 2
('dog',) 2
('on',) 2
('sat',) 2
('bone',) 1
('fish',) 1
('mat',) 1
('rug',) 1

2-gram counts:
('<s>', 'the') 4
('ate', 'the') 2
('on', 'the') 2
('sat', 'on') 2
('the', 'cat') 2
('the', 'dog') 2
('bone', '</s>') 1
('cat', 'ate') 1
('cat', 'sat') 1
('dog', 'ate') 1
('dog', 'sat') 1
('fish', '</s>') 1
('mat', '</s>') 1
('rug', '</s>') 1
('the', 'bone') 1
('the', 'fish') 1
('the', 'mat') 1
('the', 'rug') 1

3-gram counts:
('<s>', 'the', 'cat') 2
('<s>', 'the', 'dog') 2
('sat', 'on', 'the') 2
('ate', 'the', 'bone') 1
('ate', 'the', 'fish') 1
('cat', 'ate', 'the') 1
('cat', 'sat', 'on') 1
('dog', 'ate', 'the') 1
('dog', 'sat', 'on') 1
('on', 'the', 'mat') 1
('on', 'the', 'rug') 1
('the', 'bone', '</s>') 1
('the', 'cat', 'ate') 1
('the', 'cat', 'sat') 1
('the', 'dog', 'ate') 1
('the', 'dog', 'sat') 1
('the', 'fish', '</s>') 1
('the', 'mat', '</s>') 1
('the', 'rug', '</s>') 1

Top 5 bigrams: [(('<s>', 'the'), 4), (('ate', 

## Task 4 — Unigram model

In [4]:
from itertools import permutations
from math import prod

# <s> is a context marker; only words and </s> are predicted.
N = sum(count for (word,), count in counts[1].items() if word != "<s>")

def unigram_prob(w):
    return count_of((w,)) / N if w != "<s>" else 0.0

def unigram_sentence_prob(sentence):
    return prod(unigram_prob(w) for w in tokenize(sentence)[1:])

print("Word              Count    P(word)")
for word in sorted(vocab):
    print(f"{word:<17} {count_of((word,)):>3}    {unigram_prob(word):.4f}")
assert abs(sum(unigram_prob(w) for w in vocab) - 1) < 1e-12
for sentence in test:
    print(sentence, unigram_sentence_prob(sentence))

words = test[0].split()
orders = []
for order in permutations(words):
    if order not in orders:
        orders.append(order)
    if len(orders) == 2:
        break
for order in orders:
    sentence = " ".join(order)
    print(sentence, unigram_sentence_prob(sentence))
assert unigram_sentence_prob(" ".join(orders[0])) == unigram_sentence_prob(" ".join(orders[1]))

Word              Count    P(word)
</s>                4    0.1538
ate                 2    0.0769
bone                1    0.0385
cat                 2    0.0769
dog                 2    0.0769
fish                1    0.0385
mat                 1    0.0385
on                  2    0.0769
rug                 1    0.0385
sat                 2    0.0769
the                 8    0.3077
the cat sat on the rug 2.549861058867735e-07
the fish sat on the mat 1.2749305294338675e-07
the cat sat on the rug 2.549861058867735e-07
the cat sat on rug the 2.549861058867735e-07


## Task 5 — Bigram MLE

In [5]:
def bigram_prob(prev, w):
    denominator = count_of((prev,)) if prev != "</s>" else 0
    return count_of((prev, w)) / denominator if denominator else 0.0

def next_word_distribution(prev):
    return {w: bigram_prob(prev, w) for w in sorted(vocab) if count_of((prev, w))}

def bigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    return prod(bigram_prob(prev, w) for prev, w in get_ngrams(tokens, 2))

def predict_next(prev):
    distribution = next_word_distribution(prev)
    return sorted(distribution, key=lambda w: (-distribution[w], w))[0] if distribution else None

print("After 'the':", next_word_distribution("the"))
assert abs(sum(next_word_distribution("the").values()) - 1) < 1e-12
for sentence in test:
    print(sentence, bigram_sentence_prob(sentence))
for prev in ("<s>", "the", "cat", "dog", "sat", "ate", "fish"):
    print(f"After {prev}: {predict_next(prev)}")

After 'the': {'bone': 0.125, 'cat': 0.25, 'dog': 0.25, 'fish': 0.125, 'mat': 0.125, 'rug': 0.125}
the cat sat on the rug 0.015625
the fish sat on the mat 0.0
After <s>: the
After the: cat
After cat: ate
After dog: ate
After sat: on
After ate: the
After fish: </s>


## Task 6 — Trigram model

In [6]:
def trigram_prob(w1, w2, w3):
    denominator = count_of((w1, w2))
    return count_of((w1, w2, w3)) / denominator if denominator else 0.0

def trigram_sentence_prob(sentence):
    tokens = tokenize(sentence)
    first = bigram_prob(tokens[0], tokens[1])
    return first * prod(trigram_prob(*gram) for gram in get_ngrams(tokens, 3))

for sentence in test:
    print(sentence, trigram_sentence_prob(sentence))
print("N    Distinct")
for n in (1, 2, 3):
    print(f"{n}    {len(counts[n])}")
for sentence in test:
    tokens = tokenize(sentence)
    unseen_bi = [g for g in get_ngrams(tokens, 2) if not count_of(g)]
    unseen_tri = [g for g in get_ngrams(tokens, 3) if not count_of(g)]
    print("\n", sentence)
    print("Unseen bigrams:", unseen_bi)
    print("Unseen trigrams:", unseen_tri)
    winner = "trigram" if len(unseen_tri) > len(unseen_bi) else "bigram" if len(unseen_bi) > len(unseen_tri) else "tie"
    print("More unseen:", winner)

the cat sat on the rug 0.125
the fish sat on the mat 0.0
N    Distinct
1    12
2    18
3    19

 the cat sat on the rug
Unseen bigrams: []
Unseen trigrams: []
More unseen: tie

 the fish sat on the mat
Unseen bigrams: [('fish', 'sat')]
Unseen trigrams: [('<s>', 'the', 'fish'), ('the', 'fish', 'sat'), ('fish', 'sat', 'on')]
More unseen: trigram


## Task 7 — Zero probability

In [7]:
import math

def unseen_bigrams(sentence):
    return [gram for gram in get_ngrams(tokenize(sentence), 2) if not count_of(gram)]

def safe_log(p):
    return math.log(p) if p > 0 else -math.inf

for sentence in test:
    print(sentence, "unseen:", unseen_bigrams(sentence), "MLE:", bigram_sentence_prob(sentence))
assert any(bigram_sentence_prob(sentence) == 0 for sentence in test)

words_only = vocab - {"</s>"}
observed = sum(count_of((a, b)) > 0 for a in words_only for b in words_only)
possible = len(words_only) ** 2
print(f"Unseen word pairs: {possible - observed}/{possible} = {(possible - observed) / possible:.2%}")
log_prob = sum(safe_log(bigram_prob(a, b)) for a, b in get_ngrams(tokenize(test[1]), 2))
print("Second test sentence log probability:", log_prob)

the cat sat on the rug unseen: [] MLE: 0.015625
the fish sat on the mat unseen: [('fish', 'sat')] MLE: 0.0
Unseen word pairs: 87/100 = 87.00%
Second test sentence log probability: -inf


## Task 8 — Add-one and add-k smoothing

In [8]:
def bigram_prob_addk(prev, w, k):
    if k <= 0:
        raise ValueError("k must be positive")
    return (count_of((prev, w)) + k) / ((count_of((prev,)) if prev != "</s>" else 0) + k * V)

def bigram_prob_addone(prev, w):
    return bigram_prob_addk(prev, w, 1)

print("Pair            MLE       Add-one")
for prev, w in (("fish", "sat"), ("the", "cat")):
    print(f"{w} | {prev:<6} {bigram_prob(prev, w):.4f}    {bigram_prob_addone(prev, w):.4f}")
for prev in vocab | {"<s>"}:
    assert abs(sum(bigram_prob_addone(prev, w) for w in vocab) - 1) < 1e-12
print("All add-one distributions sum to 1.")
print("k       P(sat | fish)")
for k in (1, 0.5, 0.1, 0.01):
    print(f"{k:<5}   {bigram_prob_addk('fish', 'sat', k):.4f}")

Pair            MLE       Add-one
sat | fish   0.0000    0.0833
cat | the    0.2500    0.1579
All add-one distributions sum to 1.
k       P(sat | fish)
1       0.0833
0.5     0.0769
0.1     0.0476
0.01    0.0090


## Task 9 — Perplexity and model comparison

In [9]:
def perplexity(sentences, prob_fn):
    log_sum = 0.0
    predictions = 0
    for sentence in sentences:
        for prev, w in get_ngrams(tokenize(sentence), 2):
            p = prob_fn(prev, w)
            if p <= 0:
                return math.inf
            log_sum += math.log(p)
            predictions += 1
    return math.exp(-log_sum / predictions)

def unigram_prob_addone(w):
    return (count_of((w,)) + 1) / (N + V)

print("Model                  Perplexity")
for name, fn in (("Bigram MLE", bigram_prob),
                 ("Bigram add-one", bigram_prob_addone),
                 ("Unigram add-one", lambda prev, w: unigram_prob_addone(w))):
    print(f"{name:<22} {perplexity(test, fn):.4f}")

ks = (1, 0.5, 0.1, 0.01)
test_scores = {k: perplexity(test, lambda prev, w, k=k: bigram_prob_addk(prev, w, k)) for k in ks}
print("\nk       Test perplexity")
for k, score in test_scores.items():
    print(f"{k:<5}   {score:.4f}")
best_test_k = min(ks, key=lambda k: test_scores[k])
print("Best k on test:", best_test_k)

# Validation uses only the first three sentences; vocabulary still uses all four.
validation_counts = {n: Counter(gram for sentence in train[:3]
                                 for gram in get_ngrams(tokenize(sentence), n))
                     for n in (1, 2)}
def validation_addk(prev, w, k):
    return (validation_counts[2][(prev, w)] + k) / (validation_counts[1][(prev,)] + k * V)

validation_scores = {k: perplexity(train[3:4], lambda prev, w, k=k: validation_addk(prev, w, k))
                     for k in ks}
print("\nk       Validation perplexity")
for k, score in validation_scores.items():
    print(f"{k:<5}   {score:.4f}")
best_validation_k = min(ks, key=lambda k: validation_scores[k])
print("Best k from validation:", best_validation_k)
print("Test perplexity after training on all four:", test_scores[best_validation_k])
print("Test-selected k and perplexity:", best_test_k, test_scores[best_test_k])

Model                  Perplexity
Bigram MLE             inf
Bigram add-one         5.7969
Unigram add-one        9.1370

k       Test perplexity
1       5.7969
0.5     4.5703
0.1     2.9559
0.01    2.6272
Best k on test: 0.01

k       Validation perplexity
1       8.5857
0.5     8.0163
0.1     8.0260
0.01    13.1304
Best k from validation: 0.5
Test perplexity after training on all four: 4.5702700240469385
Test-selected k and perplexity: 0.01 2.6272203528005287
